# Lab 02: Gaussian beam analysis

This notebook runs the four supplied Excel profiles through the portable analysis script. Install the project `requirements.txt` and JupyterLab in the same environment first. The historical notebook uses missing TXT inputs and has different results; see [the review](../ANALYSIS_REVIEW.md).


In [ ]:
from pathlib import Path
import importlib.util
from IPython.display import display, Image, Markdown

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "projects/gaussian-beam-analysis"]
project = next((p for p in candidates if (p / "src/analyze.py").is_file()), None)
if project is None:
    raise FileNotFoundError("Open this notebook from its project folder, notebook folder, or repository root.")
spec = importlib.util.spec_from_file_location("beam_analysis", project / "src/analyze.py")
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
output = project.parents[1] / "runs/lab2"
summary = analysis.run_analysis(project / "config/analysis.json", output)


In [ ]:
lines = ["| Profile | Radius (samples) | Conditional standard error |", "| --- | ---: | ---: |"]
for record in summary["profiles"]:
    fit = record["free_baseline"]
    error = fit["radius_stderr_samples"]
    error_text = "unavailable" if error is None else f"{error:.3f}"
    lines.append(f"| {record['label']} | {fit['radius_samples']:.3f} | {error_text} |")
display(Markdown("\n".join(lines)))
display(Image(filename=str(output / "profile-fits.png")))


In [ ]:
display(Image(filename=str(output / "expander-baseline-sensitivity.png")))
print(summary["expansion"]["status"])
for mode in ["free_baseline", "zero_baseline_sensitivity"]:
    print(mode, summary["expansion"][mode])
for record in summary["profiles"]:
    for warning in record["free_baseline"]["warnings"]:
        print(record["id"], warning)


## Interpretation

The large change in the expanded-beam fit when the baseline is fixed to zero is a model limitation. Saturation, structured residuals and unverified spatial sampling prevent a reliable magnification or physical-waist claim. The original report states 3.2 µm/pixel, but the matching acquisition settings were not supplied. The code leaves physical widths unavailable.

This notebook and the portable refactor were prepared with AI assistance. See [provenance](../PROVENANCE.md).
